In [1]:
"""
Merge the three district-level flood datasets into one master table
with a composite historical flood-risk score per district.

Input:  DFSI.csv, District_FloodImpact.csv, District_FloodedArea.csv
Output: district_flood_risk_master.csv
"""
import pandas as pd
import numpy as np

DATA_DIR = "."

dfsi = pd.read_csv(f"{DATA_DIR}/DFSI.csv")
dfsi.columns = ["Dist_Name", "State_Name", "DFSI"]

impact = pd.read_csv(f"{DATA_DIR}/District_FloodImpact.csv")
area = pd.read_csv(f"{DATA_DIR}/District_FloodedArea.csv")

# normalize join key
for df in (dfsi, impact, area):
    df["Dist_Name"] = df["Dist_Name"].str.strip()

# CAVEAT: several district names repeat across different states in India
# (e.g. "Aurangabad" exists in both Bihar and Maharashtra, "Hamirpur" in both
# Uttar Pradesh and Himachal Pradesh). DFSI.csv carries State_Name so these
# are distinguishable there, but District_FloodImpact.csv / District_FloodedArea.csv
# do NOT carry a state column, so a same-named district in two states cannot be
# told apart in those two files. Left un-deduped, joining on Dist_Name alone
# fans out (18 duplicated names x their states = merge blow-up, 816 rows
# instead of ~726). We keep the highest-DFSI (worst-case) entry per duplicated
# name so the model doesn't silently double-count risk for those ~18 districts.
# This is a data-quality gap, not a computation choice -- flag it if you extend
# this dataset later with a state-tagged source.
dfsi_dedup = dfsi.sort_values("DFSI", ascending=False).drop_duplicates(
    subset="Dist_Name", keep="first"
)
# impact.csv and area.csv have the same collision (6 names each, e.g. Aurangabad,
# Balrampur, Bilaspur, Hamirpur, Pratapgarh, Raigarh) with no state column to
# disambiguate -- dedupe the same way so the join doesn't fan out.
impact_dedup = impact.drop_duplicates(subset="Dist_Name", keep="first")
area_dedup = area.drop_duplicates(subset="Dist_Name", keep="first")

master = dfsi_dedup.merge(impact_dedup, on="Dist_Name", how="inner").merge(
    area_dedup, on="Dist_Name", how="inner"
)

# one row (East, Delhi) has a missing DFSI value in the source file -- drop it
# rather than impute, since DFSI is 40% of the composite score and there's no
# safe way to guess it.
n_before = len(master)
master = master.dropna(subset=["DFSI"]).reset_index(drop=True)
if len(master) < n_before:
    print(f"Dropped {n_before - len(master)} row(s) with missing DFSI.")

# --- feature engineering ---
# fatality/injury normalized by population (raw counts are dominated by population size)
master["fatality_rate"] = master["Human_fatality"] / master["Population"] * 1e5  # per 100k
master["injury_rate"] = master["Human_injured"] / master["Population"] * 1e5
master["Mean_Flood_Duration"] = master["Mean_Flood_Duration"].fillna(
    master["Mean_Flood_Duration"].median()
)

# --- composite risk score (0-100) ---
# min-max normalize each component, then weighted blend.
# DFSI already captures severity+spread+duration historically -> highest weight.
# Corrected_Percent_Flooded_Area captures satellite-observed inundation extent.
# fatality_rate captures human-impact severity.
# Mean_Flood_Duration captures how long roads/areas stay unusable (critical for routing).
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

master["risk_score"] = (
    0.40 * minmax(master["DFSI"])
    + 0.25 * minmax(master["Corrected_Percent_Flooded_Area"])
    + 0.20 * minmax(master["fatality_rate"])
    + 0.15 * minmax(master["Mean_Flood_Duration"])
) * 100

# risk tiers for classification target (quartiles)
master["risk_tier"] = pd.qcut(
    master["risk_score"], q=4, labels=["Low", "Medium", "High", "Very High"]
)

master = master.sort_values("risk_score", ascending=False).reset_index(drop=True)
master.to_csv(f"{DATA_DIR}/district_flood_risk_master.csv", index=False)

print(f"Merged {len(master)} districts.")
print(master[["Dist_Name", "State_Name", "DFSI", "risk_score", "risk_tier"]].head(15))
print()
print("Risk tier distribution:")
print(master["risk_tier"].value_counts())

Dropped 1 row(s) with missing DFSI.
Merged 725 districts.
                    Dist_Name     State_Name       DFSI  risk_score  risk_tier
0                       Patna          BIHAR  19.300564   66.744787  Very High
1                   Begusarai          BIHAR  17.410781   66.380889  Very High
2                   Karimganj          ASSAM  17.194902   62.971575  Very High
3                       Arwal          BIHAR  14.601791   62.467986  Very High
4                    Vaishali          BIHAR  16.699429   60.738389  Very High
5                  Samastipur          BIHAR  17.761994   60.054106  Very High
6                 Murshidabad    WEST BENGAL  18.910884   59.877957  Very High
7                     Nalanda          BIHAR  16.168936   59.752242  Very High
8                      Ballia  UTTAR PRADESH  18.282413   59.188353  Very High
9                  Kendrapara         ODISHA  15.214873   58.786965  Very High
10                  Bhagalpur          BIHAR  17.604486   58.579619  Very

In [2]:
"""
Train a model on district_flood_risk_master.csv.

Why train a model at all, when risk_score is already a computed formula?
--------------------------------------------------------------------------
Because the composite score needs DFSI + flooded-area + impact data for a
district to exist. If you later want to score a NEW area that only has
partial data available live (e.g. you only have rainfall + elevation +
population density from OSM/DEM, no historical DFSI), you need a model
that has LEARNED the relationship between cheap-to-get features and the
historical risk outcome, so it can generalize/extrapolate to places
outside this 726-district table (e.g. sub-district grid cells later).

This script trains:
  1. RandomForestClassifier -> predicts risk_tier (Low/Medium/High/Very High)
  2. RandomForestRegressor  -> predicts continuous risk_score (0-100)
from the underlying raw features (NOT from DFSI itself, to avoid leakage --
DFSI is 40% of how risk_score was built, so for the "generalizes to new
areas" use case we deliberately train on the independent raw signals only:
flooded area extent, fatality/injury rates, flood duration, population).
"""
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, mean_absolute_error
import joblib

df = pd.read_csv("district_flood_risk_master.csv")

FEATURES = [
    "Percent_Flooded_Area",
    "Parmanent_Water",
    "Corrected_Percent_Flooded_Area",
    "fatality_rate",
    "injury_rate",
    "Mean_Flood_Duration",
    "Population",
]

X = df[FEATURES]
y_class = df["risk_tier"]
y_reg = df["risk_score"]

X_train, X_test, yc_train, yc_test, yr_train, yr_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

# --- classifier ---
clf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42
)
clf.fit(X_train, yc_train)
pred_c = clf.predict(X_test)
print("=== Classifier: risk_tier ===")
print(classification_report(yc_test, pred_c))
cv_scores = cross_val_score(clf, X, y_class, cv=5)
print(f"5-fold CV accuracy: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}")

# --- regressor ---
reg = RandomForestRegressor(
    n_estimators=300, max_depth=6, min_samples_leaf=5, random_state=42
)
reg.fit(X_train, yr_train)
pred_r = reg.predict(X_test)
mae = mean_absolute_error(yr_test, pred_r)
print(f"\n=== Regressor: risk_score ===")
print(f"MAE: {mae:.2f} (score range is 0-100)")

# --- feature importance ---
importances = pd.Series(clf.feature_importances_, index=FEATURES).sort_values(
    ascending=False
)
print("\n=== Feature importance (classifier) ===")
print(importances)

# --- save artifacts ---
joblib.dump(clf, "flood_risk_classifier.joblib")
joblib.dump(reg, "flood_risk_regressor.joblib")
print("\nSaved: flood_risk_classifier.joblib, flood_risk_regressor.joblib")

=== Classifier: risk_tier ===
              precision    recall  f1-score   support

        High       0.76      0.69      0.72        36
         Low       0.94      0.92      0.93        37
      Medium       0.73      0.83      0.78        36
   Very High       0.83      0.81      0.82        36

    accuracy                           0.81       145
   macro avg       0.82      0.81      0.81       145
weighted avg       0.82      0.81      0.81       145

5-fold CV accuracy: 0.745 +/- 0.116

=== Regressor: risk_score ===
MAE: 2.58 (score range is 0-100)

=== Feature importance (classifier) ===
fatality_rate                     0.224419
Corrected_Percent_Flooded_Area    0.195997
Mean_Flood_Duration               0.179977
Percent_Flooded_Area              0.154229
Population                        0.135740
injury_rate                       0.073134
Parmanent_Water                   0.036504
dtype: float64

Saved: flood_risk_classifier.joblib, flood_risk_regressor.joblib


In [3]:
"""
Shows the full inference path:
  1. Load a district's static risk (from the trained model, or straight from
     the master table if it's one of the 725 known districts).
  2. Pull live/forecast rainfall for that district from Open-Meteo (no API key).
  3. Combine them into a final "right now" risk score to feed the routing engine.

This is the piece that runs continuously in your backend -- static risk is
cheap (lookup or one model.predict() call), live weather is the part that
changes minute to minute.

NOTE: this script calls the public internet (Open-Meteo). It will not run
inside a sandboxed/offline environment -- run it on your own machine/server.
"""
import requests
import pandas as pd
import joblib

master = pd.read_csv("district_flood_risk_master.csv")
clf = joblib.load("flood_risk_classifier.joblib")
reg = joblib.load("flood_risk_regressor.joblib")

FEATURES = [
    "Percent_Flooded_Area",
    "Parmanent_Water",
    "Corrected_Percent_Flooded_Area",
    "fatality_rate",
    "injury_rate",
    "Mean_Flood_Duration",
    "Population",
]


def get_static_risk(district_name: str) -> dict:
    """Look up precomputed risk for a known district, or fall back to the
    trained model if you've built the same 7 features for a new area
    (e.g. a sub-district grid cell from OSM + DEM elevation + population)."""
    row = master[master["Dist_Name"].str.lower() == district_name.lower()]
    if len(row):
        r = row.iloc[0]
        return {"risk_score": r["risk_score"], "risk_tier": r["risk_tier"]}
    raise ValueError(
        f"'{district_name}' not in master table -- build its 7 features "
        f"and call clf.predict()/reg.predict() directly instead."
    )


def get_live_rainfall(lat: float, lon: float) -> dict:
    """Open-Meteo: no API key, no signup, free for non-commercial use."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "current": "precipitation,rain",
        "hourly": "precipitation,rain,soil_moisture_0_to_1cm",
        "forecast_days": 2,
    }
    resp = requests.get(url, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    return {
        "current_precip_mm": data["current"]["precipitation"],
        "current_rain_mm": data["current"]["rain"],
        "next_6h_precip_mm": sum(data["hourly"]["precipitation"][:6]),
    }


def combined_risk(district_name: str, lat: float, lon: float) -> dict:
    static = get_static_risk(district_name)
    weather = get_live_rainfall(lat, lon)

    # simple multiplicative rule: heavy live rain amplifies static susceptibility.
    # tune these thresholds against real events once you have ground truth.
    rain_6h = weather["next_6h_precip_mm"]
    if rain_6h < 5:
        multiplier = 1.0
    elif rain_6h < 20:
        multiplier = 1.3
    elif rain_6h < 50:
        multiplier = 1.7
    else:
        multiplier = 2.2

    live_score = min(static["risk_score"] * multiplier, 100)

    return {
        "district": district_name,
        "static_risk_score": round(static["risk_score"], 1),
        "static_risk_tier": static["risk_tier"],
        "next_6h_rain_mm": rain_6h,
        "live_risk_score": round(live_score, 1),
        "route_avoid": live_score > 60,  # feed this flag to the A*/Dijkstra edge weights
    }


if __name__ == "__main__":
    # example: Patna, Bihar (top of the static risk table)
    result = combined_risk("Patna", lat=25.5941, lon=85.1376)
    print(result)

{'district': 'Patna', 'static_risk_score': np.float64(66.7), 'static_risk_tier': 'Very High', 'next_6h_rain_mm': 6.7, 'live_risk_score': np.float64(86.8), 'route_avoid': np.True_}


In [4]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Define configuration parameters for the data generation
NUM_ROWS = 1050
districts_pool = [
    ("Mangaluru_Coastal", "KARNATAKA", "Coastal Lowland"),
    ("Jeppu_Lowlands", "KARNATAKA", "River Basin Lowland"),
    ("Kadri_Hills", "KARNATAKA", "Elevated Ridge"),
    ("Bunder_Port", "KARNATAKA", "Estuary Lowland"),
    ("Bejai_Urban", "KARNATAKA", "Mid-Elevation Plain"),
    ("Ullal_Beach", "KARNATAKA", "Coastal Lowland"),
    ("Kulshekar_Ridge", "KARNATAKA", "Elevated Ridge"),
    ("Patna_Central", "BIHAR", "Gangetic Plain"),
    ("Begusarai_East", "BIHAR", "Gangetic Plain"),
    ("Arwal_Basin", "BIHAR", "Gangetic Plain"),
    ("Vaishali_Valley", "BIHAR", "Gangetic Plain"),
    ("Samastipur_Plain", "BIHAR", "Gangetic Plain")
]

data_records = []

for i in range(NUM_ROWS):
    # Pick a baseline profile from the pool cyclically to ensure diverse regional coverage
    base_dist, state, terrain = districts_pool[i % len(districts_pool)]
    dist_name = f"{base_dist}_{i // len(districts_pool) + 1}"
    
    # Generate population matching regional configurations
    if "BIHAR" in state or "Plain" in terrain:
        population = int(np.random.randint(500000, 6000000))
    else:
        population = int(np.random.randint(40000, 750000))
        
    # Generate hydrology features based on terrain types
    if "Lowland" in terrain or "Basin" in terrain:
        permanent_water = round(np.random.uniform(2.5, 5.0), 2)
        percent_flooded = round(np.random.uniform(18.0, 45.0), 2)
        mean_duration = round(np.random.uniform(10.0, 28.0), 1)
        base_fatalities = np.random.randint(15, 400)
        base_injured = np.random.randint(5, 150)
    elif "Elevated" in terrain or "Ridge" in terrain:
        permanent_water = round(np.random.uniform(0.05, 0.5), 2)
        percent_flooded = round(np.random.uniform(0.5, 8.0), 2)
        mean_duration = round(np.random.uniform(0.5, 4.0), 1)
        base_fatalities = np.random.randint(0, 5)
        base_injured = np.random.randint(0, 10)
    else: # Mid-elevation fields
        permanent_water = round(np.random.uniform(0.5, 2.5), 2)
        percent_flooded = round(np.random.uniform(5.0, 18.0), 2)
        mean_duration = round(np.random.uniform(3.0, 12.0), 1)
        base_fatalities = np.random.randint(2, 50)
        base_injured = np.random.randint(1, 40)

    # Compute dependent structural parameters
    corrected_flooded = max(round(percent_flooded - permanent_water, 2), 0.1)
    
    # Calculate rates scaled per 100,000 population to match target schema format
    fatality_rate = round((base_fatalities / population) * 100000, 6)
    injury_rate = round((base_injured / population) * 100000, 6)
    
    # Dynamic mathematical scoring function mapping features to a clean 0-100 range
    dfsi_score = round((corrected_flooded * 0.4) + (mean_duration * 1.2) + (np.log1p(population) * 0.5), 2)
    
    # Main continuous composite risk metric formulation
    risk_score = round(
        (corrected_flooded * 1.2) + 
        (mean_duration * 1.5) + 
        (min(fatality_rate * 1.8, 30.0)) + 
        (min(injury_rate * 0.8, 10.0)), 2
    )
    
    # Ensure strict adherence to the 0-100 score boundary limit
    risk_score = min(max(risk_score, 5.0), 100.0)
    
    # Assign tiers cleanly using the balanced regression output boundaries
    if risk_score >= 60.0:
        risk_tier = "Very High"
    elif risk_score >= 45.0:
        risk_tier = "High"
    elif risk_score >= 25.0:
        risk_tier = "Medium"
    else:
        risk_tier = "Low"

    data_records.append({
        "Dist_Name": dist_name,
        "State_Name": state,
        "DFSI": dfsi_score,
        "Human_fatality": base_fatalities,
        "Human_injured": base_injured,
        "Population": population,
        "Mean_Flood_Duration": mean_duration,
        "Percent_Flooded_Area": percent_flooded,
        "Parmanent_Water": permanent_water,
        "Corrected_Percent_Flooded_Area": corrected_flooded,
        "fatality_rate": fatality_rate,
        "injury_rate": injury_rate,
        "risk_score": risk_score,
        "risk_tier": risk_tier
    })

# Convert payload to Dataframe and export to CSV
df_master = pd.DataFrame(data_records)
df_master.to_csv("district_flood_risk_master.csv", index=False)
print(f"Successfully generated a robust dataset: 'district_flood_risk_master.csv' with {len(df_master)} records.")

Successfully generated a robust dataset: 'district_flood_risk_master.csv' with 1050 records.
